# Tutorial 3: Distributed Inference Across a Constellation

**Objective:** Simulate distributed and pipeline-parallel ML inference across an orbital constellation, analyze communication overhead, and explore federated learning with bandwidth constraints.

## Background

Modern satellite constellations (Starlink, Kuiper) have **inter-satellite links (ISL)** — laser or RF connections between neighboring satellites. This enables:
- **Data-parallel inference:** Split a batch across multiple satellites
- **Pipeline-parallel inference:** Split model layers across satellites
- **Federated learning:** Train locally, aggregate globally

The challenge: ISL bandwidth is limited (~10 Gbps) and latency is non-trivial (light-speed propagation across hundreds of km). This tutorial explores these tradeoffs.

In this tutorial you will:
1. Build an ISL network from satellite positions
2. Run data-parallel inference and measure communication overhead
3. Run pipeline-parallel inference and compare
4. Simulate a federated learning round with gradient compression
5. Add ground station downlink scheduling

In [ ]:
import torch
import torch.nn as nn

from space_ml_sim.core.orbit import OrbitConfig, position_at
from space_ml_sim.core.constellation import generate_walker_delta
from space_ml_sim.environment.isl_network import ISLNetwork
from space_ml_sim.environment.ground_station import (
    GroundStation, find_contact_windows, GROUND_STATION_PRESETS
)
from space_ml_sim.compute.distributed import (
    DistributedInferenceTask, DistributedExecutor
)
from space_ml_sim.compute.model_parallel import (
    partition_model, PipelineExecutor
)
from space_ml_sim.compute.federated import FederatedCoordinator

## Step 1: Build a small constellation and ISL network

We'll create a 6-satellite Walker-Delta constellation at 550 km and build the ISL graph.

In [ ]:
# Generate constellation orbits
orbits = generate_walker_delta(
    num_satellites=6,
    num_planes=2,
    altitude_km=550,
    inclination_deg=53.0,
)

# Get positions at t=0
positions = {}
for i, orbit in enumerate(orbits):
    pos = position_at(orbit, t_seconds=0.0)
    positions[f"SAT-{i}"] = pos
    print(f"SAT-{i}: ({pos[0]:.0f}, {pos[1]:.0f}, {pos[2]:.0f}) km")

# Build ISL network (max range 5000 km, 10 Gbps per link)
network = ISLNetwork.from_positions(
    positions, max_range_km=5000.0, bandwidth_gbps=10.0
)
print(f"\nISL links established: {network.num_links}")

## Step 2: Data-parallel inference

Split a batch of inputs across worker satellites. Each worker runs the full model on its partition.

In [ ]:
def model_factory():
    return nn.Sequential(
        nn.Linear(20, 64), nn.ReLU(),
        nn.Linear(64, 32), nn.ReLU(),
        nn.Linear(32, 10),
    )

task = DistributedInferenceTask(
    model_factory=model_factory,
    num_partitions=3,
    input_shape=(12, 20),  # 12 samples split across 3 workers
    seed=42,
)

executor = DistributedExecutor(network=network)
result = executor.execute(
    task=task,
    source_node="SAT-0",
    worker_nodes=["SAT-1", "SAT-2", "SAT-3"],
)

print(f"Predictions: {result.predictions}")
print(f"Compute latency:       {result.compute_latency_ms:.2f} ms")
print(f"Communication latency: {result.communication_latency_ms:.2f} ms")
print(f"Total latency:         {result.total_latency_ms:.2f} ms")
print(f"Workers used: {result.nodes_used}")

## Step 3: Pipeline-parallel inference

Split the model's layers across satellites. Each satellite processes one stage and forwards activations.

In [ ]:
model = model_factory()
model.eval()

# Split into 3 pipeline stages
stages = partition_model(model, num_stages=3)
print(f"Pipeline stages: {len(stages)}")
for i, stage in enumerate(stages):
    params = sum(p.numel() for p in stage.parameters())
    print(f"  Stage {i}: {params:,} parameters")

pipe_executor = PipelineExecutor(network=network)
pipe_result = pipe_executor.execute(
    stages=stages,
    pipeline_nodes=["SAT-0", "SAT-1", "SAT-2"],
    input_tensor=torch.randn(12, 20),
)

print(f"\nPipeline predictions: {pipe_result.predictions}")
print(f"Compute latency:       {pipe_result.compute_latency_ms:.2f} ms")
print(f"Communication latency: {pipe_result.communication_latency_ms:.2f} ms")
print(f"Total latency:         {pipe_result.total_latency_ms:.2f} ms")
print(f"Activation sizes transferred: {pipe_result.activation_sizes_bytes} bytes")

## Step 4: Federated learning with gradient compression

Each satellite trains locally on its own data, then sends compressed gradients to an aggregator.

In [ ]:
# Simulate local datasets on each worker
torch.manual_seed(123)
datasets = {
    "SAT-1": (torch.randn(32, 20), torch.randint(0, 10, (32,))),
    "SAT-2": (torch.randn(32, 20), torch.randint(0, 10, (32,))),
    "SAT-3": (torch.randn(32, 20), torch.randint(0, 10, (32,))),
}

coord = FederatedCoordinator(
    network=network,
    aggregator_node="SAT-0",
    worker_nodes=["SAT-1", "SAT-2", "SAT-3"],
)

# Run without compression
result_full = coord.run_round(
    model_factory=model_factory,
    datasets=datasets,
    local_epochs=3,
    lr=0.01,
    compression_method="none",
)

# Run with top-k compression (keep only 10% of gradients)
result_compressed = coord.run_round(
    model_factory=model_factory,
    datasets=datasets,
    local_epochs=3,
    lr=0.01,
    compression_method="top_k",
    compression_ratio=0.1,
)

print("=== No Compression ===")
print(f"  Bytes transferred: {result_full.total_bytes_transferred:,}")
print(f"  Communication latency: {result_full.communication_latency_ms:.2f} ms")
print(f"  Worker losses: {', '.join(f'{k}: {v:.4f}' for k, v in result_full.worker_losses.items())}")

print("\n=== Top-K 10% Compression ===")
print(f"  Bytes transferred: {result_compressed.total_bytes_transferred:,}")
print(f"  Communication latency: {result_compressed.communication_latency_ms:.2f} ms")

savings = 1 - result_compressed.total_bytes_transferred / result_full.total_bytes_transferred
print(f"\nBandwidth savings: {savings:.0%}")

## Step 5: Ground station downlink scheduling

After inference, results need to be downlinked to ground. Let's find contact windows.

In [ ]:
# Use the Svalbard station (polar, good for LEO)
svalbard = GROUND_STATION_PRESETS["svalbard"]
print(f"Station: {svalbard.name} ({svalbard.latitude_deg}N, {svalbard.longitude_deg}E)")

# Find contact windows for SAT-0 over 2 orbits (~12000 seconds)
windows = find_contact_windows(
    orbit=orbits[0],
    station=svalbard,
    duration_seconds=12000.0,
    step_seconds=10.0,
)

print(f"\nContact windows over 2 orbits: {len(windows)}")
for i, w in enumerate(windows):
    downlink = w.downlink_bytes(bandwidth_gbps=1.0)  # 1 Gbps downlink
    print(f"  Window {i+1}: {w.start_seconds:.0f}s - {w.end_seconds:.0f}s "
          f"(duration: {w.duration_seconds:.0f}s, "
          f"max elev: {w.max_elevation_deg:.1f} deg, "
          f"downlink: {downlink/1e9:.2f} GB)")

## Step 6: Compare all strategies

In [ ]:
import plotly.graph_objects as go

fig = go.Figure(data=[
    go.Bar(
        name='Compute',
        x=['Data-Parallel', 'Pipeline-Parallel'],
        y=[result.compute_latency_ms, pipe_result.compute_latency_ms],
    ),
    go.Bar(
        name='Communication',
        x=['Data-Parallel', 'Pipeline-Parallel'],
        y=[result.communication_latency_ms, pipe_result.communication_latency_ms],
    ),
])

fig.update_layout(
    title='Distributed Inference: Compute vs Communication Latency',
    barmode='stack',
    yaxis_title='Latency (ms)',
    template='plotly_white',
)
fig.show()

## Exercise

1. **Increase constellation size** to 24 satellites. How does the ISL graph change? Are there disconnected components?
2. **Compare bandwidth settings:** Try 1 Gbps vs 100 Gbps ISL bandwidth. At what point does communication overhead become negligible?
3. **Multi-round federated learning:** Run 10 rounds of `coord.run_round()`, passing `global_state` from each round to the next. Plot the loss curve.
4. **Ground station network:** Use multiple stations (Svalbard + McMurdo + Fairbanks). What's the total downlink capacity per orbit?
5. **Combined scenario:** A satellite detects an anomaly, distributes inference across neighbors, and queues results for the next ground station pass. Calculate end-to-end latency.